# 📘 Session 4: Conversational Memory, Streaming & Latency

This notebook explores advanced agent mechanics:
1. **Conversational Memory:** Integrating LangGraph's `InMemorySaver` checkpointer to preserve context across multi-turn interactions using thread IDs.
2. **Streaming Invocations:** Measuring and optimizing user-perceived responsiveness (Time to First Token / TTFT) using `.stream()` vs. `.invoke()`.

## 1. Conversational Memory with `InMemorySaver`
By binding an `InMemorySaver` checkpointer to `create_agent`, the agent persists state per `thread_id` automatically.

In [3]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_ollama import ChatOllama

# Load environment variables
load_dotenv()

def get_user_info() -> str:
    """Look up information about the current user."""
    return "No user profile on file."

# Initialize local Ollama model and stateful agent
llm = ChatOllama(model="llama3.2:3b")
agent = create_agent(
    llm,
    tools=[get_user_info],
    checkpointer=InMemorySaver(),
)

# First turn: Introduce user name within thread "1"
thread_config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Sourav."}]},
    thread_config,
)["messages"][-1].content
print(response)

Hello Sourav! I'm glad to meet you. I don't have any information about you on file, so we can start fresh. How can I assist you today? Do you have a question or a problem you'd like help with?


### Turn 2: Context Retrieval
Query the agent on the same `thread_id` without re-providing the user's name in the prompt.

In [4]:
# Second turn: Verify state retention
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    thread_config,
)["messages"][-1].content
print(response)

I remember! You told me your name is Sourav. Is there something else you'd like to know or discuss?


## 2. Latency Analysis: Synchronous `.invoke()` vs. Streaming `.stream()`
Comparing blocking generation against real-time token streaming, tracking Time to First Token (TTFT).

In [8]:
import time
from langchain_ollama import ChatOllama

# Initialize the local Ollama model
llm = ChatOllama(model="llama3.2:3b", temperature=0.7)

prompt = "Explain how GPU acceleration works in 3 short sentences."

# ==========================================
# 1. WITHOUT STREAMING (.invoke)
# ==========================================
print("=== WITHOUT STREAMING ===")
start_time = time.time()
print("Generating response... (Screen is completely blank)")

# .invoke() blocks execution until ALL tokens are fully generated
response = llm.invoke(prompt)

time_to_first_word = time.time() - start_time
print(f"Time elapsed before ANY text appeared: {time_to_first_word:.2f} seconds\n")
print(response.content)
print("\n" + "="*40 + "\n")


# ==========================================
# 2. WITH STREAMING (.stream)
# ==========================================
print("=== WITH STREAMING ===")
start_time = time.time()
first_token_received = False

# .stream() yields chunks as Ollama outputs them live
for chunk in llm.stream(prompt):
    if not first_token_received:
        ttft = time.time() - start_time
        print(f"Time to First Token (TTFT): {ttft:.2f} seconds\n")
        print("Output: ", end="")
        first_token_received = True
    
    # Print each chunk instantly to stdout
    print(chunk.content, end="", flush=True)

print("\n")

=== WITHOUT STREAMING ===
Generating response... (Screen is completely blank)
Time elapsed before ANY text appeared: 2.32 seconds

GPU acceleration uses specialized graphics processing units (GPUs) to offload computationally intensive tasks from the central processing unit (CPU), allowing for faster processing of large amounts of data. By leveraging massively parallel processing and parallel computing capabilities, GPUs can perform tasks such as data compression, encryption, and scientific simulations much faster than traditional CPUs. The processed data is then transferred back to the CPU, where it can be used for final rendering or analysis.


=== WITH STREAMING ===
Time to First Token (TTFT): 0.03 seconds

Output: GPU (Graphics Processing Unit) acceleration works by utilizing the powerful parallel processing capabilities of the GPU to handle complex computational tasks. The GPU's many cores and high memory bandwidth allow it to quickly process large amounts of data, making it ideal 

## 3. Key Takeaways
- **Checkpoint Isolation:** `thread_id` allows scaling conversational memory across multiple concurrent users or tickets.
- **Perceived Latency:** While total processing time may be comparable, streaming drops TTFT to fractions of a second, drastically improving user experience.